In [ ]:
# ============================================================
# DAY 2 — CELL 1
# Mount Drive and locate frozen Day 1 artifacts
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json

# Same project root used on Day 1
PROJECT_DIR = Path("/content/drive/MyDrive/legacyagent")

print("Project directory:", PROJECT_DIR)
print("Exists:", PROJECT_DIR.exists())

# Find the important Day 1 outputs instead of guessing filenames
jsonl_files = sorted(PROJECT_DIR.rglob("*.jsonl"))
json_files = sorted(PROJECT_DIR.rglob("*.json"))

print("\nJSONL files found:")
for path in jsonl_files:
    print(" -", path.relative_to(PROJECT_DIR))

print("\nRelevant JSON files found:")
for path in json_files:
    name = path.name.lower()
    if any(key in name for key in [
        "case", "report", "qlora", "compatibility", "split"
    ]):
        print(" -", path.relative_to(PROJECT_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/legacyagent
Exists: True

JSONL files found:
 - eval/frozen_test_benchmark.jsonl
 - manifests/test_manifest.jsonl
 - manifests/train_manifest.jsonl
 - manifests/validation_manifest.jsonl
 - results/day2_baseline/qwen3_asr_base_predictions.jsonl

Relevant JSON files found:
 - case_list.json
 - dataset_report.json
 - qlora_compatibility.json
 - supreme_court_transcripts/oyez/case_summaries.json


In [ ]:
# ============================================================
# DAY 2 — CELL 8A
# Install Transformers with Qwen3-ASR support
# ============================================================

!pip -q install --upgrade \
    git+https://github.com/huggingface/transformers.git

print("Installation complete.")
print("Restart the Colab runtime before continuing.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Installation complete.
Restart the Colab runtime before continuing.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/legacyagent")

benchmark = PROJECT_DIR / "eval/frozen_test_benchmark.jsonl"

predictions = (
    PROJECT_DIR
    / "results"
    / "day2_baseline"
    / "qwen3_asr_base_predictions.jsonl"
)

print("Project folder :", PROJECT_DIR.exists())
print("Benchmark      :", benchmark.exists())
print("Predictions    :", predictions.exists())

Project folder : True
Benchmark      : True
Predictions    : True


In [ ]:
# ============================================================
# DAY 2 — CELL 2
# Load and verify the frozen Day 1 dataset split
# ============================================================

from pathlib import Path
import json

PROJECT_DIR = Path("/content/drive/MyDrive/legacyagent")
MANIFEST_DIR = PROJECT_DIR / "manifests"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_jsonl(MANIFEST_DIR / "train_manifest.jsonl")
val_rows   = load_jsonl(MANIFEST_DIR / "validation_manifest.jsonl")
test_rows  = load_jsonl(MANIFEST_DIR / "test_manifest.jsonl")

print("Raw manifest rows:")
print(f"  Train      : {len(train_rows)}")
print(f"  Validation : {len(val_rows)}")
print(f"  Test       : {len(test_rows)}")

# Inspect schema before assuming field names
print("\nManifest fields:")
print(sorted(test_rows[0].keys()))

print("\nFirst test row:")
print(json.dumps(test_rows[0], indent=2))

# Extract frozen case sets
train_cases = {row["case_id"] for row in train_rows}
val_cases   = {row["case_id"] for row in val_rows}
test_cases  = {row["case_id"] for row in test_rows}

print("\nUnique cases:")
print(f"  Train      : {len(train_cases)}")
print(f"  Validation : {len(val_cases)}")
print(f"  Test       : {len(test_cases)}")

# Hard leakage checks
assert train_cases.isdisjoint(val_cases), "❌ Train/validation leakage"
assert train_cases.isdisjoint(test_cases), "❌ Train/test leakage"
assert val_cases.isdisjoint(test_cases), "❌ Validation/test leakage"
assert len(train_cases) == 12, "❌ Expected 12 train cases"
assert len(val_cases) == 3, "❌ Expected 3 validation cases"
assert len(test_cases) == 5, "❌ Expected 5 frozen test cases"

print("\n✅ CASE-DISJOINT SPLIT VERIFIED")
print("✅ 12 train / 3 validation / 5 frozen test cases")
print("✅ Day 2 may use the test set for evaluation only")


Raw manifest rows:
  Train      : 3702
  Validation : 903
  Test       : 1537

Manifest fields:
['audio_path', 'case_id', 'case_name', 'duration', 'end', 'needs_alignment', 'section_index', 'segment_id', 'speaker_id', 'speaker_name', 'speaker_role', 'split', 'start', 'text', 'turn_index']

First test row:
{
  "segment_id": "2016_15-118_000000",
  "case_id": "2016_15-118",
  "case_name": "Hernandez v. Mesa",
  "audio_path": "/content/drive/MyDrive/legacyagent/raw_audio/2016_15-118.wav",
  "start": 0.0,
  "end": 6.895,
  "duration": 6.895,
  "text": "We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.",
  "speaker_id": "john_g_roberts_jr",
  "speaker_name": "John G. Roberts, Jr.",
  "speaker_role": "scotus_justice",
  "section_index": 0,
  "turn_index": 0,
  "split": "test",
  "needs_alignment": false
}

Unique cases:
  Train      : 12
  Validation : 3
  Test       : 5

✅ CASE-DISJOINT SPLIT VERIFIED
✅ 12 train / 3 validation / 5 frozen test cases


In [ ]:
# ============================================================
# DAY 2 — CELL 3
# Build and freeze the evaluation subset
# ============================================================

from collections import Counter

# Separate evaluation-ready rows from unresolved long segments
eval_rows = [
    row for row in test_rows
    if row["needs_alignment"] is False
]

alignment_rows = [
    row for row in test_rows
    if row["needs_alignment"] is True
]

print("Frozen test manifest:")
print(f"  Total rows              : {len(test_rows)}")
print(f"  Evaluation-ready rows   : {len(eval_rows)}")
print(f"  Needs alignment         : {len(alignment_rows)}")

# Verify all 5 frozen test cases remain represented
eval_cases = {row["case_id"] for row in eval_rows}

print(f"\nCases represented in evaluation: {len(eval_cases)}")

assert len(eval_cases) == 5, \
    "❌ Filtering removed an entire test case"

# Basic evaluation-row integrity
assert all(row["text"].strip() for row in eval_rows), \
    "❌ Empty reference transcript found"

assert all(row["duration"] > 0 for row in eval_rows), \
    "❌ Invalid duration found"

assert all(row["start"] < row["end"] for row in eval_rows), \
    "❌ Invalid timestamps found"

assert all(row["needs_alignment"] is False for row in eval_rows), \
    "❌ Unresolved alignment row entered evaluation"

# Distribution by case
case_counts = Counter(row["case_id"] for row in eval_rows)

print("\nEvaluation-ready segments per case:")
for case_id, count in sorted(case_counts.items()):
    print(f"  {case_id}: {count}")

# Duration statistics
durations = [row["duration"] for row in eval_rows]

print("\nDuration statistics:")
print(f"  Minimum : {min(durations):.2f}s")
print(f"  Maximum : {max(durations):.2f}s")
print(f"  Average : {sum(durations) / len(durations):.2f}s")
print(f"  Total   : {sum(durations) / 3600:.2f} hours")

print("\n✅ DAY 2 EVALUATION SET FROZEN")
print(f"✅ {len(eval_rows)} aligned segments")
print("✅ All 5 untouched test cases preserved")

Frozen test manifest:
  Total rows              : 1537
  Evaluation-ready rows   : 1515
  Needs alignment         : 22

Cases represented in evaluation: 5

Evaluation-ready segments per case:
  1996_96-318: 282
  2000_99-1977: 290
  2008_07-1015: 325
  2013_13-115: 329
  2016_15-118: 289

Duration statistics:
  Minimum : 0.07s
  Maximum : 29.93s
  Average : 11.31s
  Total   : 4.76 hours

✅ DAY 2 EVALUATION SET FROZEN
✅ 1515 aligned segments
✅ All 5 untouched test cases preserved


In [ ]:
# ============================================================
# DAY 2 — CELL 3B
# Audit short segments before freezing evaluation protocol
# ============================================================

short_under_3s = [
    row for row in eval_rows
    if row["duration"] < 3.0
]

tiny_under_1s = [
    row for row in eval_rows
    if row["duration"] < 1.0
]

print("Short-segment audit:")
print(f"  Under 3.0 seconds : {len(short_under_3s)}")
print(f"  Under 1.0 second  : {len(tiny_under_1s)}")

print("\nDuration buckets:")
buckets = [
    ("< 0.5s", 0.0, 0.5),
    ("0.5–1s", 0.5, 1.0),
    ("1–2s",   1.0, 2.0),
    ("2–3s",   2.0, 3.0),
    ("3–10s",  3.0, 10.0),
    ("10–20s", 10.0, 20.0),
    ("20–30s", 20.0, 30.0),
]

for label, low, high in buckets:
    count = sum(low <= row["duration"] < high for row in eval_rows)
    print(f"  {label:8s}: {count}")

print("\n10 shortest evaluation rows:")

for row in sorted(eval_rows, key=lambda x: x["duration"])[:10]:
    print("-" * 70)
    print("segment_id :", row["segment_id"])
    print("case_id    :", row["case_id"])
    print("duration   :", round(row["duration"], 3))
    print("start/end  :", row["start"], "→", row["end"])
    print("speaker    :", row["speaker_name"])
    print("text       :", repr(row["text"]))

print("\n⚠️ Audit only — no rows removed yet.")

Short-segment audit:
  Under 3.0 seconds : 352
  Under 1.0 second  : 147

Duration buckets:
  < 0.5s  : 60
  0.5–1s  : 87
  1–2s    : 116
  2–3s    : 89
  3–10s   : 371
  10–20s  : 493
  20–30s  : 299

10 shortest evaluation rows:
----------------------------------------------------------------------
segment_id : 2013_13-115_000069
case_id    : 2013_13-115
duration   : 0.072
start/end  : 663.281 → 663.353
speaker    : Ian H. Gershengorn
text       : 'Okay. Yes.'
----------------------------------------------------------------------
segment_id : 2016_15-118_000180
case_id    : 2016_15-118
duration   : 0.08
start/end  : 2296.56 → 2296.64
speaker    : Stephen G. Breyer
text       : 'No --'
----------------------------------------------------------------------
segment_id : 2016_15-118_000181
case_id    : 2016_15-118
duration   : 0.09
start/end  : 2296.64 → 2296.73
speaker    : Randolph J. Ortega
text       : '-- it --'
----------------------------------------------------------------------


In [ ]:
# ============================================================
# DAY 2 — CELL 3C
# Freeze the primary evaluation protocol BEFORE model inference
# ============================================================

MIN_EVAL_DURATION = 3.0
MAX_EVAL_DURATION = 30.0

primary_eval_rows = [
    row for row in eval_rows
    if MIN_EVAL_DURATION <= row["duration"] <= MAX_EVAL_DURATION
]

excluded_short_rows = [
    row for row in eval_rows
    if row["duration"] < MIN_EVAL_DURATION
]

primary_cases = {row["case_id"] for row in primary_eval_rows}

assert len(primary_cases) == 5, "❌ A frozen test case disappeared"
assert all(not row["needs_alignment"] for row in primary_eval_rows)
assert all(3.0 <= row["duration"] <= 30.0 for row in primary_eval_rows)

print("PRIMARY DAY 2 BENCHMARK")
print("=" * 45)
print(f"Original test rows       : {len(test_rows)}")
print(f"Needs alignment excluded : {len(alignment_rows)}")
print(f"Under 3s excluded        : {len(excluded_short_rows)}")
print(f"Primary eval segments    : {len(primary_eval_rows)}")
print(f"Frozen test cases        : {len(primary_cases)}")

print("\nPrimary segments per case:")
for case_id in sorted(primary_cases):
    count = sum(
        row["case_id"] == case_id
        for row in primary_eval_rows
    )
    print(f"  {case_id}: {count}")

total_hours = sum(
    row["duration"] for row in primary_eval_rows
) / 3600

print(f"\nTotal benchmark audio: {total_hours:.2f} hours")

print("\n🔒 PRIMARY EVALUATION PROTOCOL FROZEN")
print("🔒 Duration: 3.0s to 30.0s")
print("🔒 needs_alignment: False only")
print("🔒 Same 5 case-disjoint test cases")
print("🔒 Must be reused for every later model")

PRIMARY DAY 2 BENCHMARK
Original test rows       : 1537
Needs alignment excluded : 22
Under 3s excluded        : 352
Primary eval segments    : 1163
Frozen test cases        : 5

Primary segments per case:
  1996_96-318: 220
  2000_99-1977: 237
  2008_07-1015: 244
  2013_13-115: 248
  2016_15-118: 214

Total benchmark audio: 4.63 hours

🔒 PRIMARY EVALUATION PROTOCOL FROZEN
🔒 Duration: 3.0s to 30.0s
🔒 needs_alignment: False only
🔒 Same 5 case-disjoint test cases
🔒 Must be reused for every later model


In [ ]:
# ============================================================
# DAY 2 — CELL 4
# Freeze text normalization + WER evaluation
# ============================================================

!pip -q install jiwer

import re
import unicodedata
from jiwer import wer, process_words


def normalize_for_wer(text: str) -> str:
    """
    Frozen LegacyAgent WER normalization.

    Rules:
    - Unicode normalize
    - lowercase
    - normalize apostrophes/dashes
    - remove punctuation
    - preserve letters and numbers
    - collapse whitespace

    IMPORTANT:
    Do not modify after baseline results are produced.
    """

    text = unicodedata.normalize("NFKC", text)

    text = text.lower()

    # Normalize curly apostrophes
    text = text.replace("’", "'").replace("‘", "'")

    # Normalize transcript dashes
    text = re.sub(r"[‐-‒–—−]+", " ", text)

    # Remove punctuation, keep word characters + whitespace
    text = re.sub(r"[^\w\s']", " ", text)

    # Remove apostrophes for consistent word comparison
    # e.g. we'll -> well
    text = text.replace("'", "")

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def calculate_corpus_wer(references, hypotheses):
    """
    Corpus-level WER.

    Normalize every reference and hypothesis using the
    exact same frozen function.
    """

    normalized_refs = [
        normalize_for_wer(text)
        for text in references
    ]

    normalized_hyps = [
        normalize_for_wer(text)
        for text in hypotheses
    ]

    return wer(normalized_refs, normalized_hyps)


# ------------------------------------------------------------
# Sanity tests
# ------------------------------------------------------------

test_examples = [
    (
        "We'll hear argument in Hernandez v. Mesa.",
        "well hear argument in hernandez v mesa"
    ),
    (
        "No -- it doesn't apply.",
        "no it doesnt apply"
    ),
    (
        "Amicus curiae argued otherwise.",
        "amicus curiae argued otherwise"
    )
]

print("NORMALIZATION SANITY CHECK")
print("=" * 55)

for reference, hypothesis in test_examples:

    norm_ref = normalize_for_wer(reference)
    norm_hyp = normalize_for_wer(hypothesis)

    print("\nReference :", reference)
    print("Hypothesis:", hypothesis)
    print("Norm ref  :", norm_ref)
    print("Norm hyp  :", norm_hyp)
    print("WER       :", wer(norm_ref, norm_hyp))

# Hard sanity check
perfect_refs = [
    "Hernandez v. Mesa",
    "Amicus curiae"
]

perfect_hyps = [
    "hernandez v mesa",
    "amicus curiae"
]

assert calculate_corpus_wer(
    perfect_refs,
    perfect_hyps
) == 0.0

print("\n✅ NORMALIZATION FUNCTION PASSED")
print("✅ CORPUS WER FUNCTION PASSED")
print("🔒 This evaluator is now frozen for all V1 comparisons")

NORMALIZATION SANITY CHECK

Reference : We'll hear argument in Hernandez v. Mesa.
Hypothesis: well hear argument in hernandez v mesa
Norm ref  : well hear argument in hernandez v mesa
Norm hyp  : well hear argument in hernandez v mesa
WER       : 0.0

Reference : No -- it doesn't apply.
Hypothesis: no it doesnt apply
Norm ref  : no it doesnt apply
Norm hyp  : no it doesnt apply
WER       : 0.0

Reference : Amicus curiae argued otherwise.
Hypothesis: amicus curiae argued otherwise
Norm ref  : amicus curiae argued otherwise
Norm hyp  : amicus curiae argued otherwise
WER       : 0.0

✅ NORMALIZATION FUNCTION PASSED
✅ CORPUS WER FUNCTION PASSED
🔒 This evaluator is now frozen for all V1 comparisons


In [ ]:
# ============================================================
# DAY 2 — CELL 5
# Persist the exact frozen evaluation benchmark
# ============================================================

import json
from datetime import datetime, timezone

EVAL_DIR = PROJECT_DIR / "eval"
RESULTS_DIR = PROJECT_DIR / "results" / "day2_baseline"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_MANIFEST_PATH = EVAL_DIR / "frozen_test_benchmark.jsonl"
PROTOCOL_PATH = EVAL_DIR / "evaluation_protocol.json"

# Save the exact 1,163 rows used for every V1 comparison
with open(FROZEN_MANIFEST_PATH, "w", encoding="utf-8") as f:
    for row in primary_eval_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

protocol = {
    "project": "LegacyAgent V1",
    "protocol_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_manifest": str(MANIFEST_DIR / "test_manifest.jsonl"),

    "selection_rules": {
        "split": "test",
        "needs_alignment": False,
        "min_duration_seconds": 3.0,
        "max_duration_seconds": 30.0
    },

    "benchmark": {
        "num_segments": len(primary_eval_rows),
        "num_cases": len(primary_cases),
        "case_ids": sorted(primary_cases),
        "total_audio_hours": round(
            sum(row["duration"] for row in primary_eval_rows) / 3600,
            4
        ),
        "segment_ids": [
            row["segment_id"] for row in primary_eval_rows
        ]
    },

    "wer_normalization": {
        "unicode_normalization": "NFKC",
        "lowercase": True,
        "normalize_dashes_to_space": True,
        "remove_punctuation": True,
        "remove_apostrophes": True,
        "collapse_whitespace": True
    },

    "reuse_for": [
        "Qwen3-ASR base",
        "Whisper-small reference",
        "Qwen3-ASR contextual biasing",
        "Qwen3-ASR QLoRA fine-tuned",
        "Qwen3-ASR QLoRA quantized"
    ],

    "locked": True
}

with open(PROTOCOL_PATH, "w", encoding="utf-8") as f:
    json.dump(protocol, f, indent=2, ensure_ascii=False)

# Reload to prove persistence worked
with open(FROZEN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    reloaded_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(reloaded_rows) == 1163
assert [r["segment_id"] for r in reloaded_rows] == [
    r["segment_id"] for r in primary_eval_rows
]

print("Saved:")
print(" ", FROZEN_MANIFEST_PATH)
print(" ", PROTOCOL_PATH)

print("\nVerification:")
print(f"  Reloaded segments : {len(reloaded_rows)}")
print(f"  Frozen cases      : {len(set(r['case_id'] for r in reloaded_rows))}")
print(f"  Results directory : {RESULTS_DIR}")

print("\n🔒 EXACT BENCHMARK MEMBERSHIP SAVED")
print("🔒 EVALUATION PROTOCOL SAVED")
print("✅ Ready for base-model inference")

Saved:
  /content/drive/MyDrive/legacyagent/eval/frozen_test_benchmark.jsonl
  /content/drive/MyDrive/legacyagent/eval/evaluation_protocol.json

Verification:
  Reloaded segments : 1163
  Frozen cases      : 5
  Results directory : /content/drive/MyDrive/legacyagent/results/day2_baseline

🔒 EXACT BENCHMARK MEMBERSHIP SAVED
🔒 EVALUATION PROTOCOL SAVED
✅ Ready for base-model inference


In [ ]:
# ============================================================
# DAY 2 — CELL 6
# GPU check + Qwen3-ASR inference environment
# ============================================================

import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print(
        "GPU memory      :",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    raise RuntimeError(
        "❌ No GPU detected. In Colab: Runtime → Change runtime type → GPU"
    )

print("\nInstalling Qwen3-ASR dependencies...")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU memory      : 14.56 GB

Installing Qwen3-ASR dependencies...


In [ ]:
# ============================================================
# DAY 2 — CELL 7
# Reload frozen benchmark and test one audio segment
# ============================================================

import json
import torch
import torchaudio

FROZEN_MANIFEST_PATH = (
    PROJECT_DIR / "eval" / "frozen_test_benchmark.jsonl"
)

with open(FROZEN_MANIFEST_PATH, "r", encoding="utf-8") as f:
    benchmark_rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(benchmark_rows) == 1163

sample = benchmark_rows[0]

print("Sample benchmark row")
print("=" * 60)
print("Segment ID :", sample["segment_id"])
print("Case       :", sample["case_name"])
print("Audio path :", sample["audio_path"])
print("Start      :", sample["start"])
print("End        :", sample["end"])
print("Duration   :", sample["duration"])
print("Reference  :", sample["text"])

# Load full hearing audio
waveform, sample_rate = torchaudio.load(sample["audio_path"])

# Slice exact timestamp range
start_sample = int(sample["start"] * sample_rate)
end_sample = int(sample["end"] * sample_rate)

audio_segment = waveform[:, start_sample:end_sample]

segment_duration = audio_segment.shape[1] / sample_rate

print("\nAudio verification")
print("=" * 60)
print("Sample rate     :", sample_rate)
print("Channels        :", audio_segment.shape[0])
print("Segment samples :", audio_segment.shape[1])
print("Actual duration :", round(segment_duration, 3), "seconds")

assert sample_rate == 16000, "Expected 16 kHz audio"
assert audio_segment.shape[0] == 1, "Expected mono audio"
assert abs(segment_duration - sample["duration"]) < 0.1

print("\nAudio slicing passed")
print("Frozen benchmark is ready for model inference")

Sample benchmark row
Segment ID : 2016_15-118_000000
Case       : Hernandez v. Mesa
Audio path : /content/drive/MyDrive/legacyagent/raw_audio/2016_15-118.wav
Start      : 0.0
End        : 6.895
Duration   : 6.895
Reference  : We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.

Audio verification
Sample rate     : 16000
Channels        : 1
Segment samples : 110320
Actual duration : 6.895 seconds

Audio slicing passed
Frozen benchmark is ready for model inference


In [ ]:
# ============================================================
# DAY 2 — CELL 8
# Load base Qwen3-ASR for inference
# ============================================================

import torch
from transformers import (
    AutoProcessor,
    Qwen3ASRForConditionalGeneration
)

MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"

print("Loading processor...")
qwen_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

print("Loading base Qwen3-ASR...")

qwen_model = Qwen3ASRForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

qwen_model.eval()

print("\nModel loaded")
print("Model ID :", MODEL_ID)
print("Class    :", qwen_model.__class__.__name__)
print("Training :", qwen_model.training)
print("Device   :", next(qwen_model.parameters()).device)
print("Dtype    :", next(qwen_model.parameters()).dtype)

Loading processor...


processor_config.json:   0%|          | 0.00/487 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

...

Loading base Qwen3-ASR...


model.safetensors:   0%|          | 0.00/4.08G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]


Model loaded
Model ID : Qwen/Qwen3-ASR-1.7B-hf
Class    : Qwen3ASRForConditionalGeneration
Training : False
Device   : cuda:0
Dtype    : torch.bfloat16


In [ ]:
# ============================================================
# DAY 2 — CELL 9
# Fix Qwen input dtype and retry single-segment inference
# ============================================================

print("Input tensors before dtype fix:")
for key, value in inputs.items():
    if isinstance(value, torch.Tensor):
        print(
            f"  {key:25s}",
            "shape =", tuple(value.shape),
            "dtype =", value.dtype,
            "device =", value.device
        )

# Move tensors to GPU.
# Cast only floating-point tensors to the model dtype.
# Keep token IDs and masks as integer tensors.
fixed_inputs = {}

for key, value in inputs.items():
    if isinstance(value, torch.Tensor):

        value = value.to(qwen_model.device)

        if value.is_floating_point():
            value = value.to(torch.bfloat16)

        fixed_inputs[key] = value

    else:
        fixed_inputs[key] = value

print("\nInput tensors after dtype fix:")
for key, value in fixed_inputs.items():
    if isinstance(value, torch.Tensor):
        print(
            f"  {key:25s}",
            "dtype =", value.dtype,
            "device =", value.device
        )

with torch.inference_mode():
    generated_ids = qwen_model.generate(
        **fixed_inputs,
        max_new_tokens=128,
        do_sample=False
    )

prompt_length = fixed_inputs["input_ids"].shape[1]
generated_only = generated_ids[:, prompt_length:]

prediction = qwen_processor.batch_decode(
    generated_only,
    skip_special_tokens=True
)[0].strip()

print("\nReference:")
print(sample["text"])

print("\nQwen prediction:")
print(prediction)

single_wer = calculate_corpus_wer(
    [sample["text"]],
    [prediction]
)

print("\nSingle-segment WER:", round(single_wer, 4))

Input tensors before dtype fix:
  input_ids                 shape = (1, 105) dtype = torch.int64 device = cuda:0
  attention_mask            shape = (1, 105) dtype = torch.int64 device = cuda:0
  input_features            shape = (1, 128, 700) dtype = torch.float32 device = cuda:0
  input_features_mask       shape = (1, 700) dtype = torch.int32 device = cuda:0

Input tensors after dtype fix:
  input_ids                 dtype = torch.int64 device = cuda:0
  attention_mask            dtype = torch.int64 device = cuda:0
  input_features            dtype = torch.bfloat16 device = cuda:0
  input_features_mask       dtype = torch.int32 device = cuda:0

Reference:
We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.

Qwen prediction:
language English<asr_text>You'll hear argument first this morning in Case Fifteen One Eighteen, Hernandez versus Mesa, Mr. Hilliard.

Single-segment WER: 0.5


In [ ]:
# ============================================================
# DAY 2 — CELL 10
# Extract actual transcript from Qwen structured output
# ============================================================

import re


def parse_qwen_asr_output(raw_output: str) -> str:
    """
    Extract only the transcription text from Qwen3-ASR output.

    Example raw output:
    language English<asr_text>Hello world

    Returns:
    Hello world
    """

    text = raw_output.strip()

    # Preferred Qwen3-ASR marker
    if "<asr_text>" in text:
        text = text.split("<asr_text>", 1)[1]

    # Defensive cleanup of possible special tokens
    text = re.sub(r"<\|.*?\|>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


raw_prediction = prediction
clean_prediction = parse_qwen_asr_output(raw_prediction)

print("RAW MODEL OUTPUT")
print("=" * 60)
print(raw_prediction)

print("\nEXTRACTED TRANSCRIPT")
print("=" * 60)
print(clean_prediction)

print("\nREFERENCE")
print("=" * 60)
print(sample["text"])

corrected_single_wer = calculate_corpus_wer(
    [sample["text"]],
    [clean_prediction]
)

print("\nCorrected single-segment WER:")
print(round(corrected_single_wer, 4))

assert "language English" not in clean_prediction
assert "<asr_text>" not in clean_prediction
assert len(clean_prediction) > 0

print("\nQwen output parser passed")
print("Actual ASR transcript extraction is ready")

RAW MODEL OUTPUT
language English<asr_text>You'll hear argument first this morning in Case Fifteen One Eighteen, Hernandez versus Mesa, Mr. Hilliard.

EXTRACTED TRANSCRIPT
You'll hear argument first this morning in Case Fifteen One Eighteen, Hernandez versus Mesa, Mr. Hilliard.

REFERENCE
We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.

Corrected single-segment WER:
0.3125

Qwen output parser passed
Actual ASR transcript extraction is ready


In [ ]:
# ============================================================
# DAY 2 — RESUME CELL
# Restore parser + results directory in a fresh runtime
# ============================================================

import re

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
    / "day2_baseline"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def parse_qwen_asr_output(raw_output: str) -> str:

    text = raw_output.strip()

    if "<asr_text>" in text:
        text = text.split("<asr_text>", 1)[1]

    text = re.sub(r"<\|.*?\|>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


print("Parser restored")
print("Results directory:", RESULTS_DIR)

Parser restored
Results directory: /content/drive/MyDrive/legacyagent/results/day2_baseline


In [ ]:
# ============================================================
# DAY 2 — CELL 11
# Checkpointed full Qwen baseline inference runner
# ============================================================

import json
import time
import torch
import torchaudio
from pathlib import Path

QWEN_RESULTS_PATH = (
    RESULTS_DIR / "qwen3_asr_base_predictions.jsonl"
)

CHECKPOINT_EVERY = 10


def load_existing_predictions(path):
    """
    Resume safely after Colab disconnect.
    """

    if not path.exists():
        return [], set()

    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    completed_ids = {
        row["segment_id"]
        for row in rows
        if row.get("status") == "success"
    }

    return rows, completed_ids


def save_prediction(path, result):
    """
    Append one completed prediction immediately to Drive.
    """

    with open(path, "a", encoding="utf-8") as f:
        f.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )


def transcribe_qwen_segment(
    row,
    waveform,
    sample_rate
):
    """
    Transcribe one frozen benchmark segment.
    """

    start_sample = int(row["start"] * sample_rate)
    end_sample = int(row["end"] * sample_rate)

    audio_segment = waveform[
        :,
        start_sample:end_sample
    ]

    audio_array = (
        audio_segment
        .squeeze(0)
        .numpy()
    )

    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "audio": audio_array
                }
            ]
        }
    ]

    inputs = qwen_processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    fixed_inputs = {}

    for key, value in inputs.items():

        if isinstance(value, torch.Tensor):

            value = value.to(qwen_model.device)

            if value.is_floating_point():
                value = value.to(torch.bfloat16)

            fixed_inputs[key] = value

        else:
            fixed_inputs[key] = value

    with torch.inference_mode():

        generated_ids = qwen_model.generate(
            **fixed_inputs,
            max_new_tokens=256,
            do_sample=False
        )

    prompt_length = fixed_inputs["input_ids"].shape[1]

    generated_only = generated_ids[
        :,
        prompt_length:
    ]

    raw_output = qwen_processor.batch_decode(
        generated_only,
        skip_special_tokens=True
    )[0].strip()

    transcript = parse_qwen_asr_output(raw_output)

    return raw_output, transcript


print("Qwen inference runner created")
print("Output path:", QWEN_RESULTS_PATH)

existing_rows, completed_ids = load_existing_predictions(
    QWEN_RESULTS_PATH
)

print("Existing saved rows :", len(existing_rows))
print("Completed segments  :", len(completed_ids))
print("Remaining segments  :", len(benchmark_rows) - len(completed_ids))
print("\nNo full inference has started yet.")

Qwen inference runner created
Output path: /content/drive/MyDrive/legacyagent/results/day2_baseline/qwen3_asr_base_predictions.jsonl
Existing saved rows : 368
Completed segments  : 368
Remaining segments  : 795

No full inference has started yet.


In [ ]:
# ============================================================
# DAY 2 — CELL 12
# Five-segment mini-run through the real inference runner
# ============================================================

MINI_RUN_SIZE = 5

mini_rows = benchmark_rows[:MINI_RUN_SIZE]

# Cache full hearing audio so we do not reload the same WAV
# for every segment
audio_cache = {}

mini_results = []

print(f"Running {MINI_RUN_SIZE}-segment mini-test\n")

for index, row in enumerate(mini_rows, start=1):

    case_id = row["case_id"]

    # Load each full hearing only once
    if case_id not in audio_cache:

        print(f"Loading hearing audio: {case_id}")

        waveform, sample_rate = torchaudio.load(
            row["audio_path"]
        )

        assert sample_rate == 16000
        assert waveform.shape[0] == 1

        audio_cache[case_id] = (
            waveform,
            sample_rate
        )

    waveform, sample_rate = audio_cache[case_id]

    start_time = time.time()

    try:

        raw_output, transcript = transcribe_qwen_segment(
            row,
            waveform,
            sample_rate
        )

        elapsed = time.time() - start_time

        result = {
            "segment_id": row["segment_id"],
            "case_id": row["case_id"],
            "reference": row["text"],
            "prediction": transcript,
            "raw_output": raw_output,
            "duration": row["duration"],
            "inference_seconds": round(elapsed, 3),
            "status": "success"
        }

        mini_results.append(result)

        segment_wer = calculate_corpus_wer(
            [row["text"]],
            [transcript]
        )

        print("=" * 70)
        print(f"[{index}/{MINI_RUN_SIZE}] {row['segment_id']}")
        print("Reference :", row["text"])
        print("Prediction:", transcript)
        print("WER       :", round(segment_wer, 4))
        print("Time      :", round(elapsed, 2), "seconds")

    except Exception as error:

        print("=" * 70)
        print(f"[{index}/{MINI_RUN_SIZE}] FAILED")
        print("Segment:", row["segment_id"])
        print("Error  :", repr(error))

        raise

assert len(mini_results) == MINI_RUN_SIZE
assert all(
    row["status"] == "success"
    for row in mini_results
)

print("\nMini-run passed")
print("Successful segments:", len(mini_results))
print(
    "Average inference time:",
    round(
        sum(r["inference_seconds"] for r in mini_results)
        / len(mini_results),
        2
    ),
    "seconds per segment"
)

Running 5-segment mini-test

Loading hearing audio: 2016_15-118
[1/5] 2016_15-118_000000
Reference : We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.
Prediction: You'll hear argument first this morning in Case Fifteen One Eighteen, Hernandez versus Mesa, Mr. Hilliard.
WER       : 0.3125
Time      : 1.79 seconds
[2/5] 2016_15-118_000002
Reference : Fourth, it was one of the most fundamental rights, the right to life. Fifth, the other government involved supports -- the government of Mexico supports the claim.
Prediction: Fourth, it was one of the most fundamental rights—the right to life. Fifth, the other government involved supports the government of Mexico supports the claim.
WER       : 0.0
Time      : 1.82 seconds
[3/5] 2016_15-118_000003
Reference : So is that -- I was trying to figure out from your brief what exactly your rule is. So are all five of those necessary, in your view, for there to be a Bivens claim? Is anything else necessary?

In [ ]:
# ============================================================
# DAY 2 — CELL 13
# Full checkpointed Qwen3-ASR baseline inference
# ============================================================

import json
import time
import torch
import torchaudio

# Reload saved progress in case this is a resumed run
existing_rows, completed_ids = load_existing_predictions(
    QWEN_RESULTS_PATH
)

remaining_rows = [
    row for row in benchmark_rows
    if row["segment_id"] not in completed_ids
]

print("FULL QWEN BASELINE RUN")
print("=" * 60)
print("Total benchmark segments :", len(benchmark_rows))
print("Already completed        :", len(completed_ids))
print("Remaining                :", len(remaining_rows))
print("Output                    :", QWEN_RESULTS_PATH)

# Cache only the current hearing to avoid unnecessary RAM use
cached_case_id = None
cached_waveform = None
cached_sample_rate = None

run_start_time = time.time()
success_count = 0
failure_count = 0

for run_index, row in enumerate(remaining_rows, start=1):

    # Load a new full hearing only when the case changes
    if row["case_id"] != cached_case_id:

        cached_waveform, cached_sample_rate = torchaudio.load(
            row["audio_path"]
        )

        assert cached_sample_rate == 16000
        assert cached_waveform.shape[0] == 1

        cached_case_id = row["case_id"]

        print(
            f"\nLoaded case: {cached_case_id}"
        )

    segment_start_time = time.time()

    try:

        raw_output, transcript = transcribe_qwen_segment(
            row,
            cached_waveform,
            cached_sample_rate
        )

        elapsed = time.time() - segment_start_time

        result = {
            "segment_id": row["segment_id"],
            "case_id": row["case_id"],
            "case_name": row["case_name"],
            "start": row["start"],
            "end": row["end"],
            "duration": row["duration"],
            "reference": row["text"],
            "prediction": transcript,
            "raw_output": raw_output,
            "inference_seconds": round(elapsed, 3),
            "model": MODEL_ID,
            "status": "success"
        }

        save_prediction(
            QWEN_RESULTS_PATH,
            result
        )

        success_count += 1

    except Exception as error:

        elapsed = time.time() - segment_start_time

        result = {
            "segment_id": row["segment_id"],
            "case_id": row["case_id"],
            "reference": row["text"],
            "duration": row["duration"],
            "error": repr(error),
            "inference_seconds": round(elapsed, 3),
            "model": MODEL_ID,
            "status": "failed"
        }

        save_prediction(
            QWEN_RESULTS_PATH,
            result
        )

        failure_count += 1

        print(
            f"\nFAILED: {row['segment_id']}"
        )
        print("Error:", repr(error))

    # Progress report every 10 attempted segments
    if (
        run_index % CHECKPOINT_EVERY == 0
        or run_index == len(remaining_rows)
    ):

        total_elapsed = time.time() - run_start_time
        average_time = total_elapsed / run_index

        estimated_remaining_seconds = (
            len(remaining_rows) - run_index
        ) * average_time

        print(
            f"\nProgress: {run_index}/{len(remaining_rows)}"
        )
        print("Successful this run :", success_count)
        print("Failed this run     :", failure_count)
        print(
            "Average time        :",
            round(average_time, 2),
            "sec/segment"
        )
        print(
            "Estimated remaining :",
            round(estimated_remaining_seconds / 60, 1),
            "minutes"
        )

        # Clear unused GPU cache periodically
        torch.cuda.empty_cache()


# Final verification
saved_rows, final_completed_ids = load_existing_predictions(
    QWEN_RESULTS_PATH
)

print("\n" + "=" * 60)
print("QWEN BASELINE RUN FINISHED")
print("=" * 60)
print("Saved rows          :", len(saved_rows))
print("Successful segments :", len(final_completed_ids))
print(
    "Failed rows         :",
    sum(r.get("status") == "failed" for r in saved_rows)
)
print("Output file         :", QWEN_RESULTS_PATH)

FULL QWEN BASELINE RUN
Total benchmark segments : 1163
Already completed        : 368
Remaining                : 795
Output                    : /content/drive/MyDrive/legacyagent/results/day2_baseline/qwen3_asr_base_predictions.jsonl

Loaded case: 2000_99-1977

Progress: 10/795
Successful this run : 10
Failed this run     : 0
Average time        : 4.72 sec/segment
Estimated remaining : 61.8 minutes

Progress: 20/795
Successful this run : 20
Failed this run     : 0
Average time        : 3.41 sec/segment
Estimated remaining : 44.0 minutes

Progress: 30/795
Successful this run : 30
Failed this run     : 0
Average time        : 3.2 sec/segment
Estimated remaining : 40.8 minutes

Progress: 40/795
Successful this run : 40
Failed this run     : 0
Average time        : 3.15 sec/segment
Estimated remaining : 39.7 minutes

Progress: 50/795
Successful this run : 50
Failed this run     : 0
Average time        : 3.14 sec/segment
Estimated remaining : 39.0 minutes

Progress: 60/795
Successful this 

In [ ]:
# ============================================================
# DAY 2 — CELL 14
# Qwen baseline prediction integrity audit
# ============================================================

import json
from collections import Counter

QWEN_RESULTS_PATH = (
    PROJECT_DIR
    / "results"
    / "day2_baseline"
    / "qwen3_asr_base_predictions.jsonl"
)

# Load saved predictions
with open(QWEN_RESULTS_PATH, "r", encoding="utf-8") as f:
    qwen_predictions = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Expected benchmark membership
expected_ids = {
    row["segment_id"]
    for row in benchmark_rows
}

# Saved prediction membership
saved_ids = [
    row["segment_id"]
    for row in qwen_predictions
]

saved_id_set = set(saved_ids)

# Audit checks
duplicate_ids = [
    segment_id
    for segment_id, count in Counter(saved_ids).items()
    if count > 1
]

missing_ids = expected_ids - saved_id_set
unexpected_ids = saved_id_set - expected_ids

failed_rows = [
    row
    for row in qwen_predictions
    if row.get("status") != "success"
]

empty_predictions = [
    row["segment_id"]
    for row in qwen_predictions
    if row.get("status") == "success"
    and not row.get("prediction", "").strip()
]

print("QWEN BASELINE INTEGRITY AUDIT")
print("=" * 60)

print(f"Expected benchmark rows : {len(benchmark_rows)}")
print(f"Saved prediction rows   : {len(qwen_predictions)}")
print(f"Unique saved IDs        : {len(saved_id_set)}")
print(f"Duplicate IDs           : {len(duplicate_ids)}")
print(f"Missing benchmark IDs   : {len(missing_ids)}")
print(f"Unexpected IDs          : {len(unexpected_ids)}")
print(f"Failed rows             : {len(failed_rows)}")
print(f"Empty predictions       : {len(empty_predictions)}")

all_checks_passed = (
    len(qwen_predictions) == len(benchmark_rows)
    and len(saved_id_set) == len(expected_ids)
    and len(duplicate_ids) == 0
    and len(missing_ids) == 0
    and len(unexpected_ids) == 0
    and len(failed_rows) == 0
    and len(empty_predictions) == 0
)

print()

if all_checks_passed:
    print("QWEN BASELINE INTEGRITY AUDIT PASSED")
    print("Exact frozen benchmark membership verified")
    print("All 1,163 predictions are valid and ready for scoring")
else:
    print("QWEN BASELINE INTEGRITY AUDIT FAILED")

    if duplicate_ids:
        print("Duplicate examples:", duplicate_ids[:5])

    if missing_ids:
        print("Missing examples:", list(missing_ids)[:5])

    if unexpected_ids:
        print("Unexpected examples:", list(unexpected_ids)[:5])

    if empty_predictions:
        print("Empty prediction examples:", empty_predictions[:5])

QWEN BASELINE INTEGRITY AUDIT
Expected benchmark rows : 1163
Saved prediction rows   : 1163
Unique saved IDs        : 1163
Duplicate IDs           : 0
Missing benchmark IDs   : 0
Unexpected IDs          : 0
Failed rows             : 0
Empty predictions       : 0

QWEN BASELINE INTEGRITY AUDIT PASSED
Exact frozen benchmark membership verified
All 1,163 predictions are valid and ready for scoring


In [ ]:
# Install WER evaluation library
!pip -q install jiwer

print("jiwer installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.4 MB/s eta 0:00:00
jiwer installed


In [ ]:
# ============================================================
# DAY 2 — CELL 15
# Official Qwen3-ASR base corpus WER
# ============================================================

from jiwer import wer

# Map predictions by stable segment ID
qwen_by_id = {
    row["segment_id"]: row
    for row in qwen_predictions
}

references = []
hypotheses = []

for benchmark_row in benchmark_rows:

    segment_id = benchmark_row["segment_id"]

    reference = normalize_for_wer(
        benchmark_row["text"]
    )

    hypothesis = normalize_for_wer(
        qwen_by_id[segment_id]["prediction"]
    )

    references.append(reference)
    hypotheses.append(hypothesis)

# Official corpus-level WER
qwen_base_wer = wer(
    references,
    hypotheses
)

print("OFFICIAL QWEN3-ASR BASELINE RESULT")
print("=" * 60)

print(f"Model              : Qwen/Qwen3-ASR-1.7B-hf")
print(f"Evaluation segments: {len(references)}")
print(f"Frozen test cases  : {len(set(row['case_id'] for row in benchmark_rows))}")
print(f"Benchmark audio    : 4.63 hours")
print()
print(f"CORPUS WER         : {qwen_base_wer:.4f}")
print(f"CORPUS WER (%)     : {qwen_base_wer * 100:.2f}%")

print()
print("This is the official frozen base-model result.")
print("All later Qwen variants must use the identical benchmark and evaluator.")

OFFICIAL QWEN3-ASR BASELINE RESULT
Model              : Qwen/Qwen3-ASR-1.7B-hf
Evaluation segments: 1163
Frozen test cases  : 5
Benchmark audio    : 4.63 hours

CORPUS WER         : 0.0771
CORPUS WER (%)     : 7.71%

This is the official frozen base-model result.
All later Qwen variants must use the identical benchmark and evaluator.


In [ ]:
# ============================================================
# DAY 2 — CELL 16
# Qwen3-ASR base WER by held-out case
# ============================================================

from collections import defaultdict
from jiwer import wer

case_references = defaultdict(list)
case_hypotheses = defaultdict(list)
case_names = {}

for benchmark_row in benchmark_rows:

    segment_id = benchmark_row["segment_id"]
    case_id = benchmark_row["case_id"]

    case_names[case_id] = benchmark_row["case_name"]

    reference = normalize_for_wer(
        benchmark_row["text"]
    )

    hypothesis = normalize_for_wer(
        qwen_by_id[segment_id]["prediction"]
    )

    case_references[case_id].append(reference)
    case_hypotheses[case_id].append(hypothesis)


qwen_per_case_results = []

for case_id in sorted(case_references):

    case_wer = wer(
        case_references[case_id],
        case_hypotheses[case_id]
    )

    qwen_per_case_results.append({
        "case_id": case_id,
        "case_name": case_names[case_id],
        "segments": len(case_references[case_id]),
        "wer": case_wer
    })


print("QWEN3-ASR BASELINE — PER-CASE WER")
print("=" * 75)

for result in qwen_per_case_results:

    print(
        f"{result['case_id']:<18} "
        f"{result['segments']:>4} segments   "
        f"WER: {result['wer'] * 100:>6.2f}%   "
        f"{result['case_name']}"
    )

print("=" * 75)
print(f"Overall corpus WER: {qwen_base_wer * 100:.2f}%")

QWEN3-ASR BASELINE — PER-CASE WER
1996_96-318         220 segments   WER:   8.25%   Richardson v. McKnight
2000_99-1977        237 segments   WER:   6.19%   Saucier v. Katz
2008_07-1015        244 segments   WER:   9.44%   Ashcroft v. Iqbal
2013_13-115         248 segments   WER:   8.95%   Wood v. Moss
2016_15-118         214 segments   WER:   5.31%   Hernandez v. Mesa
Overall corpus WER: 7.71%


In [ ]:
# ============================================================
# DAY 2 — CELL 17
# Inspect highest-error Qwen baseline segments
# ============================================================

from jiwer import wer

segment_error_rows = []

for benchmark_row in benchmark_rows:

    segment_id = benchmark_row["segment_id"]

    reference_raw = benchmark_row["text"]
    prediction_raw = qwen_by_id[segment_id]["prediction"]

    reference_norm = normalize_for_wer(reference_raw)
    prediction_norm = normalize_for_wer(prediction_raw)

    segment_wer = wer(
        reference_norm,
        prediction_norm
    )

    segment_error_rows.append({
        "segment_id": segment_id,
        "case_id": benchmark_row["case_id"],
        "case_name": benchmark_row["case_name"],
        "duration": benchmark_row["duration"],
        "speaker_name": benchmark_row["speaker_name"],
        "reference": reference_raw,
        "prediction": prediction_raw,
        "segment_wer": segment_wer
    })


# Sort from highest error to lowest
segment_error_rows.sort(
    key=lambda row: row["segment_wer"],
    reverse=True
)


print("QWEN3-ASR BASELINE — HIGHEST-ERROR SEGMENTS")
print("=" * 80)

for rank, row in enumerate(segment_error_rows[:15], start=1):

    print()
    print("-" * 80)
    print(f"Rank       : {rank}")
    print(f"Segment ID : {row['segment_id']}")
    print(f"Case       : {row['case_name']}")
    print(f"Speaker    : {row['speaker_name']}")
    print(f"Duration   : {row['duration']:.2f}s")
    print(f"Segment WER: {row['segment_wer'] * 100:.2f}%")
    print()
    print("Reference :")
    print(row["reference"])
    print()
    print("Prediction:")
    print(row["prediction"])

print()
print("=" * 80)
print("Displayed the 15 highest-error benchmark segments.")
print("Audit only — no benchmark rows or predictions were modified.")

QWEN3-ASR BASELINE — HIGHEST-ERROR SEGMENTS

--------------------------------------------------------------------------------
Rank       : 1
Segment ID : 2013_13-115_000203
Case       : Wood v. Moss
Speaker    : Stephen G. Breyer
Duration   : 3.03s
Segment WER: 125.00%

Reference :
It's the latter. Okay.

Prediction:
Security, or it's the latter. It's the latter.

--------------------------------------------------------------------------------
Rank       : 2
Segment ID : 2013_13-115_000272
Case       : Wood v. Moss
Speaker    : Steven M. Wilker
Duration   : 3.84s
Segment WER: 116.67%

Reference :
The presumptive limits on depositions. They--

Prediction:
To a certain degree, they place presumptive limits on depositions.

--------------------------------------------------------------------------------
Rank       : 3
Segment ID : 2013_13-115_000082
Case       : Wood v. Moss
Speaker    : Antonin Scalia
Duration   : 5.20s
Segment WER: 92.86%

Reference :
Then don't -- just don't call it mi

In [ ]:
# ============================================================
# DAY 2 — CELL 18
# Load Whisper-small baseline
# ============================================================

import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

WHISPER_MODEL_ID = "openai/whisper-small"

print("Loading Whisper processor...")

whisper_processor = AutoProcessor.from_pretrained(
    WHISPER_MODEL_ID
)

print("Loading Whisper-small...")

whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

whisper_model = whisper_model.to("cuda")
whisper_model.eval()

print()
print("Whisper model loaded")
print("Model ID :", WHISPER_MODEL_ID)
print("Class    :", whisper_model.__class__.__name__)
print("Training :", whisper_model.training)
print("Device   :", next(whisper_model.parameters()).device)
print("Dtype    :", next(whisper_model.parameters()).dtype)

Loading Whisper processor...


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

Loading Whisper-small...


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]


Whisper model loaded
Model ID : openai/whisper-small
Class    : WhisperForConditionalGeneration
Training : False
Device   : cuda:0
Dtype    : torch.float16


In [ ]:
# ============================================================
# DAY 2 — WHISPER AUDIO HELPER
# Restore audio segment loading in fresh runtime
# ============================================================

import soundfile as sf

def load_audio_segment(row):
    audio_path = row["audio_path"]

    audio, sr = sf.read(
        audio_path,
        start=int(row["start"] * 16000),
        stop=int(row["end"] * 16000),
        dtype="float32"
    )

    return audio, sr

print("load_audio_segment() restored")

load_audio_segment() restored


In [ ]:
# ============================================================
# DAY 2 — CELL 19
# Single-segment Whisper-small inference test
# ============================================================

import time
import torch

sample_row = benchmark_rows[0]

audio, sr = load_audio_segment(sample_row)

print("Running Whisper inference on:")
print("Segment ID :", sample_row["segment_id"])
print("Duration   :", sample_row["duration"])
print("Reference  :", sample_row["text"])

# Prepare audio
inputs = whisper_processor(
    audio,
    sampling_rate=sr,
    return_tensors="pt"
)

input_features = inputs.input_features.to(
    device="cuda",
    dtype=torch.float16
)

# Run inference
start_time = time.time()

with torch.inference_mode():
    generated_ids = whisper_model.generate(
        input_features,
        language="en",
        task="transcribe"
    )

elapsed = time.time() - start_time

# Decode
whisper_prediction = whisper_processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0].strip()

# Score with the exact frozen evaluator
single_whisper_wer = wer(
    normalize_for_wer(sample_row["text"]),
    normalize_for_wer(whisper_prediction)
)

print()
print("WHISPER PREDICTION")
print("=" * 60)
print(whisper_prediction)

print()
print("REFERENCE")
print("=" * 60)
print(sample_row["text"])

print()
print(f"Single-segment WER : {single_whisper_wer:.4f}")
print(f"Inference time     : {elapsed:.2f} seconds")

Running Whisper inference on:
Segment ID : 2016_15-118_000000
Duration   : 6.895
Reference  : We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[


WHISPER PREDICTION
We'll hear argument first this morning in Case 15-118, Hernandez v. Mesa. Mr. Hilliard.

REFERENCE
We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.

Single-segment WER : 0.0625
Inference time     : 1.11 seconds


In [ ]:
# ============================================================
# DAY 2 — CELL 20
# 5-segment Whisper-small mini-test
# ============================================================

import time
from jiwer import wer

mini_rows = benchmark_rows[:5]

mini_results = []
mini_times = []

print("Running 5-segment Whisper mini-test")
print()

for i, row in enumerate(mini_rows, start=1):

    audio, sr = load_audio_segment(row)

    inputs = whisper_processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device="cuda",
        dtype=torch.float16
    )

    start_time = time.time()

    with torch.inference_mode():
        generated_ids = whisper_model.generate(
            input_features,
            language="en",
            task="transcribe"
        )

    elapsed = time.time() - start_time

    prediction = whisper_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()

    segment_wer = wer(
        normalize_for_wer(row["text"]),
        normalize_for_wer(prediction)
    )

    mini_results.append({
        "segment_id": row["segment_id"],
        "prediction": prediction,
        "wer": segment_wer
    })

    mini_times.append(elapsed)

    print("=" * 70)
    print(f"[{i}/5] {row['segment_id']}")
    print(f"Reference : {row['text']}")
    print(f"Prediction: {prediction}")
    print(f"WER       : {segment_wer:.4f}")
    print(f"Time      : {elapsed:.2f} seconds")

print()
print("Mini-run passed")
print(f"Successful segments: {len(mini_results)}")
print(
    f"Average inference time: "
    f"{sum(mini_times) / len(mini_times):.2f} seconds per segment"
)

Running 5-segment Whisper mini-test

[1/5] 2016_15-118_000000
Reference : We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.
Prediction: We'll hear argument first this morning in Case 15-118, Hernandez v. Mesa. Mr. Hilliard.
WER       : 0.0625
Time      : 0.60 seconds
[2/5] 2016_15-118_000002
Reference : Fourth, it was one of the most fundamental rights, the right to life. Fifth, the other government involved supports -- the government of Mexico supports the claim.
Prediction: Fourth, it was one of the most fundamental rights, the right to life. Fifth, the other government involved supports the government of Mexico supports the claim.
WER       : 0.0000
Time      : 1.87 seconds
[3/5] 2016_15-118_000003
Reference : So is that -- I was trying to figure out from your brief what exactly your rule is. So are all five of those necessary, in your view, for there to be a Bivens claim? Is anything else necessary? Is that exactly the rule that you want us

In [ ]:
# ============================================================
# DAY 2 — CELL 21
# Create checkpointed Whisper-small inference runner
# ============================================================

import json
import time
from pathlib import Path

WHISPER_RESULTS_DIR = (
    PROJECT_DIR
    / "results"
    / "day2_baseline"
)

WHISPER_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

WHISPER_RESULTS_PATH = (
    WHISPER_RESULTS_DIR
    / "whisper_small_predictions.jsonl"
)


def load_completed_whisper_ids():

    if not WHISPER_RESULTS_PATH.exists():
        return set()

    completed_ids = set()

    with open(
        WHISPER_RESULTS_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            row = json.loads(line)

            if row.get("status") == "success":
                completed_ids.add(
                    row["segment_id"]
                )

    return completed_ids


def run_whisper_segment(row):

    audio, sr = load_audio_segment(row)

    inputs = whisper_processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt"
    )

    input_features = (
        inputs.input_features
        .to(
            device="cuda",
            dtype=torch.float16
        )
    )

    start_time = time.time()

    with torch.inference_mode():

        generated_ids = whisper_model.generate(
            input_features,
            language="en",
            task="transcribe"
        )

    inference_time = (
        time.time() - start_time
    )

    prediction = (
        whisper_processor
        .batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0]
        .strip()
    )

    return prediction, inference_time


completed_whisper_ids = (
    load_completed_whisper_ids()
)

remaining_whisper_rows = [
    row
    for row in benchmark_rows
    if row["segment_id"]
    not in completed_whisper_ids
]


print("Whisper inference runner created")
print("Output path:", WHISPER_RESULTS_PATH)
print(
    "Already completed:",
    len(completed_whisper_ids)
)
print(
    "Remaining segments:",
    len(remaining_whisper_rows)
)

print()
print("No full Whisper inference has started yet.")

Whisper inference runner created
Output path: /content/drive/MyDrive/legacyagent/results/day2_baseline/whisper_small_predictions.jsonl
Already completed: 0
Remaining segments: 1163

No full Whisper inference has started yet.


In [ ]:
# ============================================================
# DAY 2 — CELL 22
# Full checkpointed Whisper-small baseline inference
# ============================================================

import json
import time

# Re-check saved progress every time this cell starts
completed_whisper_ids = load_completed_whisper_ids()

remaining_whisper_rows = [
    row
    for row in benchmark_rows
    if row["segment_id"] not in completed_whisper_ids
]

total_remaining = len(remaining_whisper_rows)

print("FULL WHISPER-SMALL BASELINE RUN")
print("=" * 60)
print(f"Total benchmark segments : {len(benchmark_rows)}")
print(f"Already completed        : {len(completed_whisper_ids)}")
print(f"Remaining                : {total_remaining}")
print(f"Output                   : {WHISPER_RESULTS_PATH}")
print()

if total_remaining == 0:

    print("Whisper baseline is already complete.")

else:

    successful_this_run = 0
    failed_this_run = 0
    run_start_time = time.time()

    with open(
        WHISPER_RESULTS_PATH,
        "a",
        encoding="utf-8"
    ) as output_file:

        for index, row in enumerate(
            remaining_whisper_rows,
            start=1
        ):

            try:

                prediction, inference_time = (
                    run_whisper_segment(row)
                )

                result = {
                    "segment_id": row["segment_id"],
                    "case_id": row["case_id"],
                    "case_name": row["case_name"],
                    "reference": row["text"],
                    "prediction": prediction,
                    "duration": row["duration"],
                    "inference_time": inference_time,
                    "model_id": WHISPER_MODEL_ID,
                    "status": "success"
                }

                successful_this_run += 1

            except Exception as error:

                result = {
                    "segment_id": row["segment_id"],
                    "case_id": row["case_id"],
                    "case_name": row["case_name"],
                    "reference": row["text"],
                    "prediction": "",
                    "duration": row["duration"],
                    "model_id": WHISPER_MODEL_ID,
                    "status": "failed",
                    "error": str(error)
                }

                failed_this_run += 1

            output_file.write(
                json.dumps(
                    result,
                    ensure_ascii=False
                )
                + "\n"
            )

            # Save immediately to Google Drive
            output_file.flush()

            if index % 10 == 0 or index == total_remaining:

                elapsed = time.time() - run_start_time
                average_time = elapsed / index

                estimated_remaining = (
                    average_time
                    * (total_remaining - index)
                    / 60
                )

                print(
                    f"Progress: {index}/{total_remaining}"
                )
                print(
                    f"Successful this run : "
                    f"{successful_this_run}"
                )
                print(
                    f"Failed this run     : "
                    f"{failed_this_run}"
                )
                print(
                    f"Average time        : "
                    f"{average_time:.2f} sec/segment"
                )
                print(
                    f"Estimated remaining : "
                    f"{estimated_remaining:.1f} minutes"
                )
                print()

print("Run stopped or completed.")
print(
    "Saved successful IDs:",
    len(load_completed_whisper_ids())
)

FULL WHISPER-SMALL BASELINE RUN
Total benchmark segments : 1163
Already completed        : 0
Remaining                : 1163
Output                   : /content/drive/MyDrive/legacyagent/results/day2_baseline/whisper_small_predictions.jsonl

Progress: 10/1163
Successful this run : 10
Failed this run     : 0
Average time        : 0.60 sec/segment
Estimated remaining : 11.5 minutes

Progress: 20/1163
Successful this run : 20
Failed this run     : 0
Average time        : 0.59 sec/segment
Estimated remaining : 11.3 minutes

Progress: 30/1163
Successful this run : 30
Failed this run     : 0
Average time        : 0.56 sec/segment
Estimated remaining : 10.6 minutes

Progress: 40/1163
Successful this run : 40
Failed this run     : 0
Average time        : 0.58 sec/segment
Estimated remaining : 10.9 minutes

Progress: 50/1163
Successful this run : 50
Failed this run     : 0
Average time        : 0.60 sec/segment
Estimated remaining : 11.2 minutes

Progress: 60/1163
Successful this run : 60
Faile

In [ ]:
# ============================================================
# DAY 2 — CELL 23
# Whisper-small prediction integrity audit
# ============================================================

import json
from collections import Counter

# Load saved Whisper predictions
with open(WHISPER_RESULTS_PATH, "r", encoding="utf-8") as f:
    whisper_predictions = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Expected frozen benchmark membership
expected_ids = {
    row["segment_id"]
    for row in benchmark_rows
}

# Saved prediction membership
saved_ids = [
    row["segment_id"]
    for row in whisper_predictions
]

saved_id_set = set(saved_ids)

# Audit checks
duplicate_ids = [
    segment_id
    for segment_id, count in Counter(saved_ids).items()
    if count > 1
]

missing_ids = expected_ids - saved_id_set
unexpected_ids = saved_id_set - expected_ids

failed_rows = [
    row
    for row in whisper_predictions
    if row.get("status") != "success"
]

empty_predictions = [
    row["segment_id"]
    for row in whisper_predictions
    if row.get("status") == "success"
    and not row.get("prediction", "").strip()
]

print("WHISPER BASELINE INTEGRITY AUDIT")
print("=" * 60)

print(f"Expected benchmark rows : {len(benchmark_rows)}")
print(f"Saved prediction rows   : {len(whisper_predictions)}")
print(f"Unique saved IDs        : {len(saved_id_set)}")
print(f"Duplicate IDs           : {len(duplicate_ids)}")
print(f"Missing benchmark IDs   : {len(missing_ids)}")
print(f"Unexpected IDs          : {len(unexpected_ids)}")
print(f"Failed rows             : {len(failed_rows)}")
print(f"Empty predictions       : {len(empty_predictions)}")

all_checks_passed = (
    len(whisper_predictions) == len(benchmark_rows)
    and len(saved_id_set) == len(expected_ids)
    and len(duplicate_ids) == 0
    and len(missing_ids) == 0
    and len(unexpected_ids) == 0
    and len(failed_rows) == 0
    and len(empty_predictions) == 0
)

print()

if all_checks_passed:
    print("WHISPER BASELINE INTEGRITY AUDIT PASSED")
    print("Exact frozen benchmark membership verified")
    print("All 1,163 predictions are valid and ready for scoring")
else:
    print("WHISPER BASELINE INTEGRITY AUDIT FAILED")

    if duplicate_ids:
        print("Duplicate examples:", duplicate_ids[:5])

    if missing_ids:
        print("Missing examples:", list(missing_ids)[:5])

    if unexpected_ids:
        print("Unexpected examples:", list(unexpected_ids)[:5])

    if failed_rows:
        print(
            "Failed examples:",
            [row["segment_id"] for row in failed_rows[:5]]
        )

    if empty_predictions:
        print("Empty prediction examples:", empty_predictions[:5])

WHISPER BASELINE INTEGRITY AUDIT
Expected benchmark rows : 1163
Saved prediction rows   : 1163
Unique saved IDs        : 1163
Duplicate IDs           : 0
Missing benchmark IDs   : 0
Unexpected IDs          : 0
Failed rows             : 0
Empty predictions       : 0

WHISPER BASELINE INTEGRITY AUDIT PASSED
Exact frozen benchmark membership verified
All 1,163 predictions are valid and ready for scoring


In [ ]:
# ============================================================
# DAY 2 — CELL 24
# Official Whisper-small corpus WER
# ============================================================

from jiwer import wer

# Map predictions by stable segment ID
whisper_by_id = {
    row["segment_id"]: row
    for row in whisper_predictions
}

references = []
whisper_hypotheses = []

for benchmark_row in benchmark_rows:

    segment_id = benchmark_row["segment_id"]

    reference = normalize_for_wer(
        benchmark_row["text"]
    )

    hypothesis = normalize_for_wer(
        whisper_by_id[segment_id]["prediction"]
    )

    references.append(reference)
    whisper_hypotheses.append(hypothesis)

# Official corpus-level WER
whisper_base_wer = wer(
    references,
    whisper_hypotheses
)

print("OFFICIAL WHISPER-SMALL BASELINE RESULT")
print("=" * 60)

print(f"Model              : {WHISPER_MODEL_ID}")
print(f"Evaluation segments: {len(references)}")
print(
    f"Frozen test cases  : "
    f"{len(set(row['case_id'] for row in benchmark_rows))}"
)
print("Benchmark audio    : 4.63 hours")
print()
print(f"CORPUS WER         : {whisper_base_wer:.4f}")
print(f"CORPUS WER (%)     : {whisper_base_wer * 100:.2f}%")

print()
print("This is the official frozen Whisper-small baseline result.")
print("Direct comparison uses the identical benchmark and evaluator.")

OFFICIAL WHISPER-SMALL BASELINE RESULT
Model              : openai/whisper-small
Evaluation segments: 1163
Frozen test cases  : 5
Benchmark audio    : 4.63 hours

CORPUS WER         : 0.0985
CORPUS WER (%)     : 9.85%

This is the official frozen Whisper-small baseline result.
Direct comparison uses the identical benchmark and evaluator.


In [ ]:
# ============================================================
# DAY 2 — CELL 25
# Qwen vs Whisper per-case baseline comparison
# ============================================================

from collections import defaultdict
from jiwer import wer

case_refs = defaultdict(list)
case_qwen_hyps = defaultdict(list)
case_whisper_hyps = defaultdict(list)
case_names = {}

for row in benchmark_rows:

    segment_id = row["segment_id"]
    case_id = row["case_id"]

    case_names[case_id] = row["case_name"]

    reference = normalize_for_wer(
        row["text"]
    )

    qwen_hypothesis = normalize_for_wer(
        qwen_by_id[segment_id]["prediction"]
    )

    whisper_hypothesis = normalize_for_wer(
        whisper_by_id[segment_id]["prediction"]
    )

    case_refs[case_id].append(reference)
    case_qwen_hyps[case_id].append(qwen_hypothesis)
    case_whisper_hyps[case_id].append(whisper_hypothesis)


baseline_comparison = []

for case_id in sorted(case_refs):

    qwen_case_wer = wer(
        case_refs[case_id],
        case_qwen_hyps[case_id]
    )

    whisper_case_wer = wer(
        case_refs[case_id],
        case_whisper_hyps[case_id]
    )

    winner = (
        "Qwen"
        if qwen_case_wer < whisper_case_wer
        else "Whisper"
    )

    baseline_comparison.append({
        "case_id": case_id,
        "case_name": case_names[case_id],
        "segments": len(case_refs[case_id]),
        "qwen_wer": qwen_case_wer,
        "whisper_wer": whisper_case_wer,
        "winner": winner
    })


print("LEGACYAGENT V1 — BASELINE COMPARISON BY CASE")
print("=" * 100)

print(
    f"{'Case':<18}"
    f"{'Segments':>10}"
    f"{'Qwen WER':>14}"
    f"{'Whisper WER':>16}"
    f"{'Winner':>12}"
)

print("-" * 100)

for result in baseline_comparison:

    print(
        f"{result['case_id']:<18}"
        f"{result['segments']:>10}"
        f"{result['qwen_wer'] * 100:>13.2f}%"
        f"{result['whisper_wer'] * 100:>15.2f}%"
        f"{result['winner']:>12}"
    )

print("-" * 100)

print(
    f"{'OVERALL':<18}"
    f"{len(benchmark_rows):>10}"
    f"{qwen_base_wer * 100:>13.2f}%"
    f"{whisper_base_wer * 100:>15.2f}%"
)

absolute_improvement = (
    whisper_base_wer - qwen_base_wer
)

relative_error_reduction = (
    absolute_improvement / whisper_base_wer
)

qwen_case_wins = sum(
    result["winner"] == "Qwen"
    for result in baseline_comparison
)

print()
print(
    f"Qwen case wins             : "
    f"{qwen_case_wins}/{len(baseline_comparison)}"
)
print(
    f"Absolute WER improvement   : "
    f"{absolute_improvement * 100:.2f} points"
)
print(
    f"Relative error reduction   : "
    f"{relative_error_reduction * 100:.2f}%"
)

LEGACYAGENT V1 — BASELINE COMPARISON BY CASE
Case                Segments      Qwen WER     Whisper WER      Winner
----------------------------------------------------------------------------------------------------
1996_96-318              220         8.25%           9.01%        Qwen
2000_99-1977             237         6.19%           7.39%        Qwen
2008_07-1015             244         9.44%          11.27%        Qwen
2013_13-115              248         8.95%          11.26%        Qwen
2016_15-118              214         5.31%          10.29%        Qwen
----------------------------------------------------------------------------------------------------
OVERALL                 1163         7.71%           9.85%

Qwen case wins             : 5/5
Absolute WER improvement   : 2.14 points
Relative error reduction   : 21.71%


In [ ]:
# ============================================================
# DAY 2 — CELL 26
# Save official Day 2 baseline results
# ============================================================

import json
from datetime import datetime, timezone

BASELINE_SUMMARY_PATH = (
    PROJECT_DIR
    / "results"
    / "day2_baseline"
    / "baseline_summary.json"
)

baseline_summary = {
    "project": "LegacyAgent V1",
    "stage": "Day 2 baseline evaluation",

    "evaluation_protocol": {
        "benchmark_file": str(
            PROJECT_DIR
            / "eval"
            / "frozen_test_benchmark.jsonl"
        ),
        "evaluation_segments": len(benchmark_rows),
        "frozen_test_cases": len(
            set(row["case_id"] for row in benchmark_rows)
        ),
        "benchmark_audio_hours": 4.63,
        "minimum_duration_seconds": 3.0,
        "maximum_duration_seconds": 30.0,
        "needs_alignment": False,
        "split_policy": "case-disjoint",
        "metric": "corpus WER",
        "normalization": "frozen Day 2 normalize_for_wer"
    },

    "models": {
        "qwen3_asr_base": {
            "model_id": "Qwen/Qwen3-ASR-1.7B-hf",
            "prediction_file": str(QWEN_RESULTS_PATH),
            "prediction_rows": len(qwen_predictions),
            "integrity_audit": "passed",
            "corpus_wer": qwen_base_wer,
            "corpus_wer_percent": qwen_base_wer * 100
        },

        "whisper_small": {
            "model_id": WHISPER_MODEL_ID,
            "prediction_file": str(WHISPER_RESULTS_PATH),
            "prediction_rows": len(whisper_predictions),
            "integrity_audit": "passed",
            "corpus_wer": whisper_base_wer,
            "corpus_wer_percent": whisper_base_wer * 100
        }
    },

    "comparison": {
        "qwen_case_wins": qwen_case_wins,
        "total_test_cases": len(baseline_comparison),
        "absolute_wer_improvement_points":
            absolute_improvement * 100,
        "relative_error_reduction_percent":
            relative_error_reduction * 100
    },

    "per_case_results": baseline_comparison,

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}


with open(
    BASELINE_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        baseline_summary,
        f,
        indent=2,
        ensure_ascii=False
    )


# Reload from disk to verify the saved artifact
with open(
    BASELINE_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    verified_summary = json.load(f)


print("DAY 2 BASELINE SUMMARY SAVED")
print("=" * 60)

print("Path:", BASELINE_SUMMARY_PATH)
print()
print(
    "Evaluation segments :",
    verified_summary[
        "evaluation_protocol"
    ][
        "evaluation_segments"
    ]
)
print(
    "Qwen WER            :",
    f"{verified_summary['models']['qwen3_asr_base']['corpus_wer_percent']:.2f}%"
)
print(
    "Whisper WER         :",
    f"{verified_summary['models']['whisper_small']['corpus_wer_percent']:.2f}%"
)
print(
    "Qwen case wins      :",
    f"{verified_summary['comparison']['qwen_case_wins']}/"
    f"{verified_summary['comparison']['total_test_cases']}"
)

print()
print("Official baseline artifact verified.")

DAY 2 BASELINE SUMMARY SAVED
Path: /content/drive/MyDrive/legacyagent/results/day2_baseline/baseline_summary.json

Evaluation segments : 1163
Qwen WER            : 7.71%
Whisper WER         : 9.85%
Qwen case wins      : 5/5

Official baseline artifact verified.


In [ ]:
# ============================================================
# DAY 2 — CELL 27
# Build and freeze legal entity evaluation vocabulary
# ============================================================

import json
import re
from pathlib import Path
from collections import Counter

ENTITY_TERMS_PATH = (
    PROJECT_DIR
    / "eval"
    / "entity_terms.json"
)

# ------------------------------------------------------------
# 1. Fixed legal terms declared before model-error inspection
# ------------------------------------------------------------

starter_legal_terms = [
    "certiorari",
    "amicus curiae",
    "stare decisis",
    "sua sponte"
]


# ------------------------------------------------------------
# 2. Case names from frozen benchmark metadata
# ------------------------------------------------------------

case_names = sorted({
    row["case_name"].strip()
    for row in benchmark_rows
    if row.get("case_name", "").strip()
})


# ------------------------------------------------------------
# 3. Speaker surnames from frozen benchmark metadata
# ------------------------------------------------------------

speaker_surnames = set()

for row in benchmark_rows:

    speaker_name = row.get(
        "speaker_name",
        ""
    ).strip()

    if not speaker_name:
        continue

    if speaker_name.lower() == "unknown":
        continue

    # Remove suffixes such as Jr., Sr., II, III
    cleaned_name = re.sub(
        r",?\s+(Jr\.?|Sr\.?|II|III|IV)$",
        "",
        speaker_name,
        flags=re.IGNORECASE
    )

    parts = cleaned_name.split()

    if parts:
        surname = parts[-1].strip("., ")

        if len(surname) >= 2:
            speaker_surnames.add(surname)

speaker_surnames = sorted(
    speaker_surnames
)


# ------------------------------------------------------------
# 4. Count exact normalized occurrences in reference text
# ------------------------------------------------------------

normalized_references = [
    normalize_for_wer(row["text"])
    for row in benchmark_rows
]


def count_term_occurrences(term):

    normalized_term = normalize_for_wer(term)

    if not normalized_term:
        return 0

    pattern = re.compile(
        r"(?<!\w)"
        + re.escape(normalized_term)
        + r"(?!\w)"
    )

    return sum(
        len(pattern.findall(reference))
        for reference in normalized_references
    )


candidate_terms = []

for term in starter_legal_terms:
    candidate_terms.append({
        "term": term,
        "category": "legal_term",
        "source": "roadmap_starter_list"
    })

for term in case_names:
    candidate_terms.append({
        "term": term,
        "category": "case_name",
        "source": "frozen_benchmark_metadata"
    })

for term in speaker_surnames:
    candidate_terms.append({
        "term": term,
        "category": "speaker_surname",
        "source": "frozen_benchmark_metadata"
    })


# ------------------------------------------------------------
# 5. Keep only terms that actually occur in references
# ------------------------------------------------------------

entity_terms = []

seen_normalized_terms = set()

for item in candidate_terms:

    normalized_term = normalize_for_wer(
        item["term"]
    )

    if normalized_term in seen_normalized_terms:
        continue

    occurrence_count = count_term_occurrences(
        item["term"]
    )

    if occurrence_count == 0:
        continue

    seen_normalized_terms.add(
        normalized_term
    )

    entity_terms.append({
        **item,
        "normalized_term": normalized_term,
        "reference_occurrences": occurrence_count
    })


entity_terms.sort(
    key=lambda item: (
        item["category"],
        item["normalized_term"]
    )
)


# ------------------------------------------------------------
# 6. Save frozen evaluation artifact
# ------------------------------------------------------------

entity_eval_protocol = {
    "project": "LegacyAgent V1",
    "purpose": "Day 2 entity-span evaluation only",

    "important_rule": (
        "This vocabulary is an evaluation artifact. "
        "It must not automatically become the Day 3 bias glossary."
    ),

    "sources": [
        "roadmap starter legal terms",
        "frozen benchmark case metadata",
        "frozen benchmark speaker metadata"
    ],

    "model_error_outputs_used": False,

    "benchmark_segments": len(
        benchmark_rows
    ),

    "terms": entity_terms
}


ENTITY_TERMS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    ENTITY_TERMS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        entity_eval_protocol,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

category_counts = Counter(
    item["category"]
    for item in entity_terms
)

total_occurrences = sum(
    item["reference_occurrences"]
    for item in entity_terms
)


print("LEGAL ENTITY EVALUATION VOCABULARY FROZEN")
print("=" * 65)

print("Saved to:", ENTITY_TERMS_PATH)
print()
print("Total unique terms       :", len(entity_terms))
print("Total reference matches  :", total_occurrences)

print()
print("Terms by category:")

for category, count in sorted(
    category_counts.items()
):
    print(
        f"  {category:<20}: {count}"
    )

print()
print("Matched evaluation terms:")
print("-" * 65)

for item in entity_terms:

    print(
        f"{item['category']:<20} "
        f"{item['term']:<30} "
        f"{item['reference_occurrences']:>4} occurrences"
    )

print()
print("No Qwen or Whisper errors were used to build this vocabulary.")
print("Entity evaluation vocabulary is now frozen.")

LEGAL ENTITY EVALUATION VOCABULARY FROZEN
Saved to: /content/drive/MyDrive/legacyagent/eval/entity_terms.json

Total unique terms       : 24
Total reference matches  : 159

Terms by category:
  case_name           : 2
  legal_term          : 1
  speaker_surname     : 21

Matched evaluation terms:
-----------------------------------------------------------------
case_name            Hernandez v. Mesa                 1 occurrences
case_name            Wood v. Moss                      1 occurrences
legal_term           certiorari                        1 occurrences
speaker_surname      Alito                             4 occurrences
speaker_surname      Boyd                              4 occurrences
speaker_surname      Breyer                           24 occurrences
speaker_surname      Clement                           8 occurrences
speaker_surname      Garre                            15 occurrences
speaker_surname      Gershengorn                       5 occurrences
speaker_surname

In [ ]:
# ============================================================
# DAY 2 — CELL 28
# Audit frozen legal-entity reference matches
# ============================================================

import re
from collections import Counter, defaultdict

# Load frozen terms from the saved artifact
with open(
    ENTITY_TERMS_PATH,
    "r",
    encoding="utf-8"
) as f:
    frozen_entity_protocol = json.load(f)

frozen_entity_terms = frozen_entity_protocol["terms"]

entity_match_rows = []

for row in benchmark_rows:

    reference_norm = normalize_for_wer(
        row["text"]
    )

    for entity in frozen_entity_terms:

        term_norm = entity["normalized_term"]

        pattern = re.compile(
            r"(?<!\w)"
            + re.escape(term_norm)
            + r"(?!\w)"
        )

        matches = list(
            pattern.finditer(reference_norm)
        )

        for match_index, match in enumerate(matches):

            entity_match_rows.append({
                "segment_id": row["segment_id"],
                "case_id": row["case_id"],
                "case_name": row["case_name"],
                "term": entity["term"],
                "normalized_term": term_norm,
                "category": entity["category"],
                "reference": row["text"],
                "normalized_reference": reference_norm,
                "match_index": match_index
            })


print("FROZEN ENTITY MATCH AUDIT")
print("=" * 70)

print(
    "Expected occurrences :",
    sum(
        item["reference_occurrences"]
        for item in frozen_entity_terms
    )
)

print(
    "Recovered occurrences:",
    len(entity_match_rows)
)

print(
    "Unique segments      :",
    len({
        row["segment_id"]
        for row in entity_match_rows
    })
)

print()

category_occurrences = Counter(
    row["category"]
    for row in entity_match_rows
)

print("Occurrences by category:")

for category, count in sorted(
    category_occurrences.items()
):
    print(
        f"  {category:<20}: {count}"
    )


print()
print("Sample matched occurrences")
print("=" * 70)

samples_by_category = defaultdict(list)

for row in entity_match_rows:

    if len(samples_by_category[row["category"]]) < 5:
        samples_by_category[row["category"]].append(row)


for category in sorted(samples_by_category):

    print()
    print(category.upper())
    print("-" * 70)

    for row in samples_by_category[category]:

        print(
            f"Term       : {row['term']}"
        )
        print(
            f"Segment ID : {row['segment_id']}"
        )
        print(
            f"Reference  : {row['reference']}"
        )
        print()


match_audit_passed = (
    len(entity_match_rows)
    ==
    sum(
        item["reference_occurrences"]
        for item in frozen_entity_terms
    )
)

print("=" * 70)

if match_audit_passed:
    print("ENTITY MATCH AUDIT PASSED")
    print("All 159 frozen reference occurrences were recovered.")
else:
    print("ENTITY MATCH AUDIT FAILED")

FROZEN ENTITY MATCH AUDIT
Expected occurrences : 159
Recovered occurrences: 159
Unique segments      : 142

Occurrences by category:
  case_name           : 2
  legal_term          : 1
  speaker_surname     : 156

Sample matched occurrences

CASE_NAME
----------------------------------------------------------------------
Term       : Hernandez v. Mesa
Segment ID : 2016_15-118_000000
Reference  : We'll hear argument first this morning in Case No. 15-118, Hernandez v. Mesa. Mr. Hilliard.

Term       : Wood v. Moss
Segment ID : 2013_13-115_000000
Reference  : We'll hear argument this morning in Case 13-115, Wood v. Moss. Mr. Gershengorn.


LEGAL_TERM
----------------------------------------------------------------------
Term       : certiorari
Segment ID : 2000_99-1977_000105
Reference  : Mr. Clement, your... you raise... the Government raises two questions in its petition for certiorari and the second one is did the Court of Appeals err in concluding on the basis facts noted that the def

In [ ]:
# ============================================================
# DAY 2 — CELL 29
# Frozen entity-token error rate for Qwen and Whisper
# ============================================================

from jiwer import process_words

# ------------------------------------------------------------
# Metric definition
# ------------------------------------------------------------
#
# 1. Normalize reference and hypothesis with the frozen Day 2
#    normalization.
# 2. Find words belonging to the 159 frozen entity occurrences.
# 3. Globally align reference and hypothesis words.
# 4. Count substitutions and deletions affecting entity words.
# 5. Divide entity-word errors by total reference entity words.
#
# Important:
# Insertions are not assigned to entity spans because an inserted
# hypothesis word has no unique reference entity position.
#
# Therefore the precise metric name is:
# ENTITY-TOKEN ERROR RATE (ETER)
#
# We will save this as the project's entity-focused metric rather
# than mislabeling it as unrestricted corpus WER.
# ------------------------------------------------------------


def get_entity_word_positions(reference_text, frozen_terms):

    reference_norm = normalize_for_wer(reference_text)
    reference_words = reference_norm.split()

    entity_positions = set()

    for entity in frozen_terms:

        term_words = (
            entity["normalized_term"].split()
        )

        term_length = len(term_words)

        for start in range(
            len(reference_words)
            - term_length
            + 1
        ):

            end = start + term_length

            if (
                reference_words[start:end]
                == term_words
            ):

                entity_positions.update(
                    range(start, end)
                )

    return reference_norm, entity_positions


def calculate_entity_token_errors(
    benchmark_rows,
    predictions_by_id,
    frozen_terms
):

    total_entity_words = 0
    substitutions = 0
    deletions = 0

    evaluated_occurrence_segments = 0

    for row in benchmark_rows:

        segment_id = row["segment_id"]

        reference_norm, entity_positions = (
            get_entity_word_positions(
                row["text"],
                frozen_terms
            )
        )

        if not entity_positions:
            continue

        evaluated_occurrence_segments += 1

        hypothesis_norm = normalize_for_wer(
            predictions_by_id[
                segment_id
            ]["prediction"]
        )

        alignment = process_words(
            reference_norm,
            hypothesis_norm
        )

        total_entity_words += len(
            entity_positions
        )

        # One sentence pair is passed each time,
        # so use the first alignment list.
        for chunk in alignment.alignments[0]:

            ref_positions = range(
                chunk.ref_start_idx,
                chunk.ref_end_idx
            )

            affected_entity_words = sum(
                position in entity_positions
                for position in ref_positions
            )

            if chunk.type == "substitute":
                substitutions += (
                    affected_entity_words
                )

            elif chunk.type == "delete":
                deletions += (
                    affected_entity_words
                )

    total_errors = (
        substitutions
        + deletions
    )

    error_rate = (
        total_errors / total_entity_words
        if total_entity_words > 0
        else 0.0
    )

    return {
        "entity_words": total_entity_words,
        "evaluated_segments":
            evaluated_occurrence_segments,
        "substitutions": substitutions,
        "deletions": deletions,
        "total_errors": total_errors,
        "error_rate": error_rate
    }


qwen_entity_result = (
    calculate_entity_token_errors(
        benchmark_rows,
        qwen_by_id,
        frozen_entity_terms
    )
)

whisper_entity_result = (
    calculate_entity_token_errors(
        benchmark_rows,
        whisper_by_id,
        frozen_entity_terms
    )
)


print("LEGACYAGENT V1 — FROZEN ENTITY-FOCUSED EVALUATION")
print("=" * 70)

print("Frozen entity occurrences :", len(entity_match_rows))
print(
    "Segments with entities    :",
    len({
        row["segment_id"]
        for row in entity_match_rows
    })
)
print(
    "Reference entity words    :",
    qwen_entity_result["entity_words"]
)

print()
print("QWEN3-ASR-1.7B")
print("-" * 70)
print(
    "Entity substitutions :",
    qwen_entity_result["substitutions"]
)
print(
    "Entity deletions     :",
    qwen_entity_result["deletions"]
)
print(
    "Total entity errors  :",
    qwen_entity_result["total_errors"]
)
print(
    "Entity-token error   :",
    f"{qwen_entity_result['error_rate'] * 100:.2f}%"
)

print()
print("WHISPER-SMALL")
print("-" * 70)
print(
    "Entity substitutions :",
    whisper_entity_result["substitutions"]
)
print(
    "Entity deletions     :",
    whisper_entity_result["deletions"]
)
print(
    "Total entity errors  :",
    whisper_entity_result["total_errors"]
)
print(
    "Entity-token error   :",
    f"{whisper_entity_result['error_rate'] * 100:.2f}%"
)

entity_absolute_improvement = (
    whisper_entity_result["error_rate"]
    - qwen_entity_result["error_rate"]
)

entity_relative_reduction = (
    entity_absolute_improvement
    / whisper_entity_result["error_rate"]
    if whisper_entity_result["error_rate"] > 0
    else 0.0
)

print()
print("=" * 70)
print(
    "Absolute improvement :",
    f"{entity_absolute_improvement * 100:.2f} points"
)
print(
    "Relative reduction   :",
    f"{entity_relative_reduction * 100:.2f}%"
)

print()
print("Metric: frozen entity-token error rate (ETER)")
print("Insertions are excluded because they have no unique reference entity position.")

LEGACYAGENT V1 — FROZEN ENTITY-FOCUSED EVALUATION
Frozen entity occurrences : 159
Segments with entities    : 142
Reference entity words    : 163

QWEN3-ASR-1.7B
----------------------------------------------------------------------
Entity substitutions : 30
Entity deletions     : 20
Total entity errors  : 50
Entity-token error   : 30.67%

WHISPER-SMALL
----------------------------------------------------------------------
Entity substitutions : 35
Entity deletions     : 23
Total entity errors  : 58
Entity-token error   : 35.58%

Absolute improvement : 4.91 points
Relative reduction   : 13.79%

Metric: frozen entity-token error rate (ETER)
Insertions are excluded because they have no unique reference entity position.


In [ ]:
# ============================================================
# DAY 2 — CELL 30
# Save final Day 2 results with entity-focused evaluation
# ============================================================

import json
from datetime import datetime, timezone

# Reload the official baseline artifact
with open(
    BASELINE_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:
    final_day2_summary = json.load(f)


# ------------------------------------------------------------
# Add frozen entity evaluation protocol
# ------------------------------------------------------------

final_day2_summary["entity_evaluation"] = {
    "metric_name": "Entity-Token Error Rate",
    "metric_abbreviation": "ETER",

    "entity_terms_file": str(
        ENTITY_TERMS_PATH
    ),

    "frozen_entity_occurrences": len(
        entity_match_rows
    ),

    "segments_with_entities": len({
        row["segment_id"]
        for row in entity_match_rows
    }),

    "reference_entity_words":
        qwen_entity_result["entity_words"],

    "insertions_included": False,

    "insertion_policy": (
        "Excluded because inserted hypothesis words "
        "have no unique reference entity position."
    ),

    "vocabulary_limitations": {
        "unique_terms": len(
            frozen_entity_terms
        ),
        "speaker_surname_occurrences": 156,
        "case_name_occurrences": 2,
        "legal_term_occurrences": 1,
        "note": (
            "Targeted entity benchmark dominated by "
            "speaker surnames; not a comprehensive "
            "legal-jargon benchmark."
        )
    }
}


# ------------------------------------------------------------
# Add model entity results
# ------------------------------------------------------------

final_day2_summary[
    "models"
][
    "qwen3_asr_base"
][
    "entity_token_error_rate"
] = qwen_entity_result["error_rate"]

final_day2_summary[
    "models"
][
    "qwen3_asr_base"
][
    "entity_token_error_rate_percent"
] = (
    qwen_entity_result["error_rate"] * 100
)

final_day2_summary[
    "models"
][
    "qwen3_asr_base"
][
    "entity_substitutions"
] = qwen_entity_result["substitutions"]

final_day2_summary[
    "models"
][
    "qwen3_asr_base"
][
    "entity_deletions"
] = qwen_entity_result["deletions"]


final_day2_summary[
    "models"
][
    "whisper_small"
][
    "entity_token_error_rate"
] = whisper_entity_result["error_rate"]

final_day2_summary[
    "models"
][
    "whisper_small"
][
    "entity_token_error_rate_percent"
] = (
    whisper_entity_result["error_rate"] * 100
)

final_day2_summary[
    "models"
][
    "whisper_small"
][
    "entity_substitutions"
] = whisper_entity_result["substitutions"]

final_day2_summary[
    "models"
][
    "whisper_small"
][
    "entity_deletions"
] = whisper_entity_result["deletions"]


# ------------------------------------------------------------
# Add entity comparison
# ------------------------------------------------------------

final_day2_summary[
    "comparison"
][
    "entity_absolute_improvement_points"
] = (
    entity_absolute_improvement * 100
)

final_day2_summary[
    "comparison"
][
    "entity_relative_error_reduction_percent"
] = (
    entity_relative_reduction * 100
)


# ------------------------------------------------------------
# Mark Day 2 complete
# ------------------------------------------------------------

final_day2_summary["day2_status"] = "complete"

final_day2_summary["completed_at_utc"] = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


# ------------------------------------------------------------
# Save updated artifact
# ------------------------------------------------------------

with open(
    BASELINE_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_day2_summary,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# Reload and verify
# ------------------------------------------------------------

with open(
    BASELINE_SUMMARY_PATH,
    "r",
    encoding="utf-8"
) as f:

    verified_final_summary = json.load(f)


print("FINAL DAY 2 RESULTS SAVED")
print("=" * 65)

print("Path:", BASELINE_SUMMARY_PATH)

print()
print("Qwen3-ASR-1.7B")
print(
    "  Overall WER :",
    f"{verified_final_summary['models']['qwen3_asr_base']['corpus_wer_percent']:.2f}%"
)
print(
    "  Entity ETER :",
    f"{verified_final_summary['models']['qwen3_asr_base']['entity_token_error_rate_percent']:.2f}%"
)

print()
print("Whisper-small")
print(
    "  Overall WER :",
    f"{verified_final_summary['models']['whisper_small']['corpus_wer_percent']:.2f}%"
)
print(
    "  Entity ETER :",
    f"{verified_final_summary['models']['whisper_small']['entity_token_error_rate_percent']:.2f}%"
)

print()
print(
    "Qwen overall relative reduction :",
    f"{verified_final_summary['comparison']['relative_error_reduction_percent']:.2f}%"
)
print(
    "Qwen entity relative reduction  :",
    f"{verified_final_summary['comparison']['entity_relative_error_reduction_percent']:.2f}%"
)

print()
print(
    "Day 2 status:",
    verified_final_summary["day2_status"].upper()
)

print()
print("Official Day 2 artifact verified.")

FINAL DAY 2 RESULTS SAVED
Path: /content/drive/MyDrive/legacyagent/results/day2_baseline/baseline_summary.json

Qwen3-ASR-1.7B
  Overall WER : 7.71%
  Entity ETER : 30.67%

Whisper-small
  Overall WER : 9.85%
  Entity ETER : 35.58%

Qwen overall relative reduction : 21.71%
Qwen entity relative reduction  : 13.79%

Day 2 status: COMPLETE

Official Day 2 artifact verified.


In [37]:
import json
from pathlib import Path
from datetime import datetime, timezone

SUMMARY_PATH = Path(
    "/content/drive/MyDrive/legacyagent/results/day2_baseline/baseline_summary.json"
)

# Load the existing official summary
with open(SUMMARY_PATH, "r") as f:
    summary = json.load(f)

# Add methodological clarification only.
# NO scores, predictions, benchmark rows, or frozen protocols are changed.
summary["reporting_notes"] = {
    "benchmark_scope": (
        "Results apply to the frozen case-disjoint benchmark of 1,163 "
        "segments with durations from 3.0 to 30.0 seconds and "
        "needs_alignment=false; they are not results on complete raw hearings."
    ),
    "primary_wer_status": (
        "Official frozen Day 2 metric. Number-format variants are not "
        "canonicalized beyond the frozen normalize_for_wer protocol."
    ),
    "entity_metric_scope": (
        "ETER is measured on the frozen entity-token subset. The subset is "
        "dominated by speaker surnames and must not be described as a "
        "comprehensive legal-jargon benchmark."
    ),
    "entity_metric_interpretation": (
        "Treat the approximately 5 percentage-point Qwen advantage as a "
        "directional signal on the frozen entity subset; report the underlying "
        "reference entity-word count alongside percentages."
    ),
    "test_set_governance": (
        "The benchmark, entity vocabulary, normalization protocol, and cached "
        "predictions remain unchanged. No post-hoc test-set tuning was performed."
    )
}

summary["day2_status"] = "FINAL_FROZEN_WITH_REPORTING_CLARIFICATION"
summary["reporting_clarification_added_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

# Save back to the same official artifact
with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 70)
print("DAY 2 REPORTING CLARIFICATION SAVED")
print("=" * 70)
print("Path:", SUMMARY_PATH)
print()
print("Day 2 status:", summary["day2_status"])
print("Qwen WER remains:",
      f'{summary["models"]["qwen3_asr_base"]["corpus_wer_percent"]:.2f}%')
print("Whisper WER remains:",
      f'{summary["models"]["whisper_small"]["corpus_wer_percent"]:.2f}%')
print("Qwen ETER remains:",
      f'{summary["models"]["qwen3_asr_base"]["entity_token_error_rate_percent"]:.2f}%')
print()
print("NO inference rerun")
print("NO metric changed")
print("NO benchmark changed")
print("NO entity vocabulary changed")
print("NO normalization changed")
print("=" * 70)

DAY 2 REPORTING CLARIFICATION SAVED
Path: /content/drive/MyDrive/legacyagent/results/day2_baseline/baseline_summary.json

Day 2 status: FINAL_FROZEN_WITH_REPORTING_CLARIFICATION
Qwen WER remains: 7.71%
Whisper WER remains: 9.85%
Qwen ETER remains: 30.67%

NO inference rerun
NO metric changed
NO benchmark changed
NO entity vocabulary changed
NO normalization changed
